In [1]:
pip install -r requirements.txt

  Using cached chromadb-0.5.11-py3-none-any.whl.metadata (6.8 kB)
  Using cached langchain_chroma-0.1.4-py3-none-any.whl.metadata (1.6 kB)
  Using cached langchain_core-0.2.24-py3-none-any.whl.metadata (6.2 kB)
  Using cached langsmith-0.1.93-py3-none-any.whl.metadata (13 kB)
Using cached chromadb-0.5.11-py3-none-any.whl (603 kB)
Using cached langchain_core-0.2.24-py3-none-any.whl (377 kB)
Using cached langsmith-0.1.93-py3-none-any.whl (139 kB)
Using cached langchain_chroma-0.1.4-py3-none-any.whl (10 kB)

  Attempting uninstall: langsmith

    Found existing installation: langsmith 0.6.7

   ---------------------------------------- 0/4 [langsmith]
    Uninstalling langsmith-0.6.7:
   ---------------------------------------- 0/4 [langsmith]
      Successfully uninstalled langsmith-0.6.7
   ---------------------------------------- 0/4 [langsmith]
   ---------------------------------------- 0/4 [langsmith]
   ---------------------------------------- 0/4 [langsmith]
   --------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.1 requires langchain-core<2.0.0,>=1.2.5, but you have langchain-core 0.2.24 which is incompatible.
langchain-classic 1.0.1 requires langchain-text-splitters<2.0.0,>=1.1.0, but you have langchain-text-splitters 0.2.2 which is incompatible.
langgraph-checkpoint 4.0.0 requires langchain-core>=0.2.38, but you have langchain-core 0.2.24 which is incompatible.
langgraph-prebuilt 1.0.7 requires langchain-core>=1.0.0, but you have langchain-core 0.2.24 which is incompatible.


In [5]:
import os
from dotenv import load_dotenv
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [2]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

DATA_PATH = "data/lex"  # folder containing ALL PDFs

loader = DirectoryLoader(
    DATA_PATH,
    glob="*.pdf",
    loader_cls=PyPDFLoader
)

documents = loader.load()
print(f"Loaded {len(documents)} pages")


Loaded 801 pages


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=300
)

chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")


Created 3159 chunks


In [6]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

PERSIST_DIR = r"C:\Users\sanja\Desktop\LEX-PROJECT\chroma_db"
COLLECTION_NAME = "indian_laws"   # human-readable, no hashes

embedding = OpenAIEmbeddings()

vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory=PERSIST_DIR,
    collection_name=COLLECTION_NAME
)

vectordb.persist()
print("✅ Clean ingestion completed")


✅ Clean ingestion completed


C:\Users\sanja\Desktop\LEX-PROJECT\venv\lib\site-packages\langchain_core\_api\deprecation.py:139: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  warn_deprecated(


In [7]:
print("Vector count:", vectordb._collection.count())


Vector count: 3159


In [8]:
vectordb.similarity_search("fundamental rights", k=3)
vectordb.similarity_search("punishment for murder", k=3)


[Document(metadata={'page': 33, 'source': 'data\\lex\\bns.pdf'}, page_content='circumstances that, if he by that act caused death, he would be guilty of murder, shall be\npunished with imprisonment of either description for a term which may extend to ten years,\nand shall also be liable to fine; and if hurt is caused to any person by such act, the offender\nshall be liable either to imprisonment for life, or to such punishment as is hereinbefore\nmentioned.\nCulpable\nhomicide by\ncausing death\nof person other\nthan person\nwhose death\nwas intended.\nPunishment\nfor murder.\nPunishment\nfor murder by\nlife-convict.\nPunishment\nfor culpable\nhomicide not\namounting to\nmurder.\nCausing death\nby negligence.\nAbetment of\nsuicide of child\nor person of\nunsound mind.\nAbetment of\nsuicide.\nAttempt to\nmurder.\n30 of 2019.'),
 Document(metadata={'page': 7, 'source': 'data\\lex\\bns.pdf'}, page_content='to kill him, and not having been subject to the provocation, assists A in killing Z